# M3 decision reports on Kaggle

Runs `serpentguard.decision.decision_report` on every training run whose kernel output is attached to this notebook, on
the CPU. For each run it fits the verdict thresholds on the calibration outputs, reports the verdicts on the
validation and unknown-pool outputs, and writes `decision_metrics.json`. A summary table of the main verdict numbers
goes to `decision_summary.csv`.

`SCORER` picks what ranks the species, the species head (`"head"`) or the nearest prototype (`"prototype"`,
plan section 2.1), for the fit and every verdict; the notebook can be run once with each. With `PREDS_KERNEL`, a
run that reads predicted masks or boxes (`mask_source: predicted`, F1/F1m/F2/F3) gets `tau_snake` from the front
end's box confidences, so its report counts the photos with no snake found.

It runs on Kaggle rather than locally so the saved outputs (`*.npz`) never have to be downloaded. A kernel can read
only its own account's private outputs, so each account runs its own copy. No GPU is used.

In [1]:
RUN_KERNELS = ['puneetbajajiitm/sg-m3-fe12', 'puneetbajajiitm/sg-m3-fe34']
QUALITY_KERNEL = None   # optional "owner/slug" whose output holds quality.csv (Task 10), for the poor_image reason
PREDS_KERNEL = 'puneetbajajiitm/sg-m3-frontends'
SCORER = 'head'

In [2]:
import subprocess
import sys

SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt, a Global Constraint: every metric uses it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, f"scikit-learn {sklearn.__version__} is loaded"

In [3]:
import json
import shutil
from pathlib import Path

import kagglehub
import pandas as pd

OWNER = "puneetbajajiitm"
CODE = Path("/kaggle/input/serpentguard-code/src")
sys.path.insert(0, str(CODE if CODE.exists() else Path("../src").resolve()))

from serpentguard.config import load_config
from serpentguard.decision import decision_report
from serpentguard.run_data import FRONTEND_PREDS_FILE, PREDICTED

RUN_FILES = ("val_outputs.npz", "calib_outputs.npz", "unknown_outputs.npz", "prototypes.npz", "final_metrics.json",
             "config.yaml")
OUT = Path("/kaggle/working/decision")

data_root = Path(kagglehub.dataset_download(f"{OWNER}/serpentguard-indian-snakes-clean"))
classes = pd.read_csv(data_root / "classes.csv")
metadata = pd.read_csv(data_root / "metadata.csv", usecols=["path", "split"])


def kernel_output(handle):
    return Path(kagglehub.notebook_output_download(handle))


def run_folders(root):
    return sorted(p.parent for p in root.rglob("val_outputs.npz") if (p.parent / "calib_outputs.npz").exists())


quality = None
if QUALITY_KERNEL:
    found = sorted(kernel_output(QUALITY_KERNEL).rglob("quality.csv"))
    quality = pd.read_csv(found[0]) if found else None
    print("quality table:", found[0] if found else "not found")

box_conf = None
if PREDS_KERNEL:
    preds_files = sorted(kernel_output(PREDS_KERNEL).rglob(FRONTEND_PREDS_FILE))
    assert len(preds_files) == 1, f"{PREDS_KERNEL}'s output must hold one {FRONTEND_PREDS_FILE}; found {preds_files}"
    preds = pd.read_csv(preds_files[0], usecols=["path", "box_conf"])
    box_conf = dict(zip(preds["path"], preds["box_conf"]))
    print(f"front-end predictions: {preds_files[0]}, {preds['box_conf'].notna().sum():,} photos with a box")

front-end predictions: /kaggle/input/sg-m3-frontends/frontend_preds/frontend_preds.csv, 39,514 photos with a box


In [4]:
rows = []
for handle in RUN_KERNELS:
    for source in run_folders(kernel_output(handle)):
        run = OUT / source.name
        run.mkdir(parents=True, exist_ok=True)
        for name in RUN_FILES:
            if (source / name).exists():
                shutil.copy2(source / name, run / name)
        predicted = load_config(run / "config.yaml").mask_source == PREDICTED
        if predicted and box_conf is None:
            print(f"{source.name} reads predicted masks or boxes; without PREDS_KERNEL it gets no tau_snake")
        report = decision_report(run, classes, metadata=metadata, quality=quality, scorer=SCORER,
                                 box_conf=box_conf if predicted else None)
        for name in RUN_FILES:
            if name.endswith(".npz"):
                (run / name).unlink()
        metrics, fit = report["verdict_metrics"], report["fit"]
        rows.append({"run": source.name, "kernel": handle, "scorer": report["scorer"],
                     "who_dangerous_recall": metrics["who_dangerous_recall"], "false_alarm": metrics["false_alarm"],
                     "venomous_recall": metrics["venomous_recall"],
                     "o3_all": metrics["o3"]["dangerous_as_not_listed_all"], "abstain_rate": metrics["abstain_rate"],
                     "unknown_flagged": report["u1"]["groups"]["all"]["flagged_at_tau"],
                     "unknown_auroc_msp": report["u1"]["groups"]["all"]["open_set"]["msp"]["auroc"],
                     "theta_d": report["thresholds"]["theta_d"], "theta_who": report["thresholds"]["theta_who"],
                     "calibration_recall": fit["who_dangerous_recall"],
                     "tau_snake": report["no_snake"].get("tau_snake"), "no_snake_val": report["no_snake"]["val"]})
        print(f"{source.name}: WHO recall {metrics['who_dangerous_recall']:.3f}, false alarms "
              f"{metrics['false_alarm']:.3f}, unknown flagged {rows[-1]['unknown_flagged']:.3f}")
summary = pd.DataFrame(rows)
summary.to_csv(OUT / "decision_summary.csv", index=False)
assert len(summary), "no run folders found in the attached kernel outputs"
summary

FE_F1: WHO recall 0.922, false alarms 0.279, unknown flagged 0.078
FE_F1m: WHO recall 0.932, false alarms 0.301, unknown flagged 0.075
FE_F2: WHO recall 0.937, false alarms 0.328, unknown flagged 0.064
FE_F3: WHO recall 0.954, false alarms 0.416, unknown flagged 0.075


,run,kernel,scorer,who_dangerous_recall,false_alarm,venomous_recall,o3_all,abstain_rate,unknown_flagged,unknown_auroc_msp,theta_d,theta_who,calibration_recall,tau_snake,no_snake_val
0,FE_F1,puneetbajajiitm/sg-m3-fe12,head,0.922102,0.278522,0.922858,0.035195,0.119643,0.077605,0.680069,0.101804,inf,0.950530,0.102,151
1,FE_F1m,puneetbajajiitm/sg-m3-fe12,head,0.932426,0.301249,0.931918,0.029564,0.101568,0.075388,0.663243,0.081300,inf,0.950530,0.102,151
2,FE_F2,puneetbajajiitm/sg-m3-fe34,head,0.936649,0.327815,0.931400,0.033318,0.094805,0.064302,0.663918,0.085952,inf,0.952297,0.102,151
3,FE_F3,puneetbajajiitm/sg-m3-fe34,head,0.954481,0.416466,0.944085,0.024871,0.061279,0.075388,0.637777,0.066737,inf,0.957597,0.102,151
